# 第 10 章:LoRA —— 从零实现参数高效微调

第 9 章我们学了全参 SFT(Supervised Fine-Tuning)—— 更新模型的**每一个参数**来让它学会新能力。

但全参 SFT 有个问题:**太贵了**。一个 64M 的模型要存全量梯度、全量 optimizer state,显存翻 3-4 倍。换一个领域就要重新训一遍,存一整套权重。

LoRA(Low-Rank Adaptation)给出了一条更聪明的路:**冻结原始权重,只训练一个极小的「补丁」**。

本章从零手写一个 naive LoRA layer,再展示 minimind 的 monkey-patch 实现,最后走完训练 → 保存 → 合并的完整流程。

> 本章对应 minimind 的两个核心文件:
> - `model/model_lora.py`(65 行,全文):LoRA 模块 + apply / save / load / merge
> - `trainer/train_lora.py`(186 行):冻结非 LoRA 参数,只训 LoRA

## 环境准备

导入 minimind 组件,构造一个标准 64M 模型。本章所有代码基于这个模型实例。

注意:这里用**随机初始化**(不加载预训练权重)—— 因为本章关心的是**LoRA 的结构和参数量**,不是训练效果。

In [ ]:
import sys, torch, torch.nn as nn
sys.path.insert(0, '/home/minimind')

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config = MiniMindConfig()
model = MiniMindForCausalLM(config).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"hidden_size      = {config.hidden_size}")            # 768
print(f"num_hidden_layers = {config.num_hidden_layers}")       # 8
print(f"num_attention_heads = {config.num_attention_heads}")   # 8
print(f"num_key_value_heads = {config.num_key_value_heads}")   # 4 (GQA)
print(f"head_dim          = {config.head_dim}")                # 96
print(f"vocab_size        = {config.vocab_size}")              # 6400
print(f"\n模型总参数量: {total_params:,} ({total_params/1e6:.2f}M)")

&nbsp;

---

## 10.1 为什么需要参数高效微调

全参 SFT 的流程是:加载预训练权重 → 在新数据上训练 → **所有参数都被更新**。

这在概念上很直接,但工程上代价巨大:

| 问题 | 全参 SFT | 说明 |
|---|---|---|
| 显存 | 需要 optimizer state + 梯度 | AdamW 存 2 份 state(momentum + variance),每份和模型一样大 |
| 存储 | 每个领域存一整套权重 | 训 3 个领域 = 存 3 份 64M 的 `.pth` |
| 切换 | 必须重新加载整个模型 | 换领域 = 换文件 + 重载 |
| 训练速度 | 更新所有参数 | backward 要对全量参数算梯度 |

LoRA 的洞察:**微调时的权重变化 $\Delta W$ 是「低秩」的** —— 不需要 $d \times d$ 的完整矩阵,一个 $d \times r$ 和 $r \times d$ 的乘积就够了($r \ll d$)。

用代码感受一下这个参数量差距:

In [ ]:
d = 768  # hidden_size

# 假设我们想微调一个 768×768 的权重矩阵
full_params = d * d
print(f"=== 全参微调一个 768×768 层 ===")
print(f"参数量: {full_params:,}")

# LoRA 用 rank=16 的低秩分解
r = 16
lora_params = d * r + r * d  # A + B
print(f"\n=== LoRA rank={r} 微调同一个层 ===")
print(f"A: {d}×{r} = {d*r:,}")
print(f"B: {r}×{d} = {r*d:,}")
print(f"合计: {lora_params:,}")
print(f"压缩比: {full_params / lora_params:.1f}×")
print(f"LoRA 占比: {lora_params / full_params * 100:.2f}%")

一个 768×768 的层,全参需要 589,824 个参数,LoRA(rank=16)只要 24,576 个 —— **少了 24 倍**。

而且 LoRA 的关键优势在于:**原始权重 $W$ 被冻结**,只有 $A$ 和 $B$ 需要梯度、optimizer state。存储时只存 $A+B$,切换领域时只需加载一个小文件。

> 这就是 LoRA 的核心价值:用 ~0.6% 的可训练参数,达到接近全参微调的效果。

### 为什么低秩假设成立?

LoRA 的理论基础来自一个经验观察:**预训练模型在微调时,权重的变化 $\Delta W$ 具有很低的「内在秩」(intrinsic rank)**。

换句话说,虽然 $\Delta W$ 形式上是 $d \times d$ 的矩阵,但它真正「有效」的信息只集中在少数几个方向上。用 $d \times r$ 和 $r \times d$ 的两个小矩阵就能近似它 —— 这就是「低秩近似」。

LoRA 论文(2021,微软)实验表明:即使 $r=1$ 或 $r=2$,LoRA 在很多任务上就能接近全参微调的效果。minimind 用 $r=16$,是一个非常安全的选择。

> 直觉:微调不是「重新学习」,而是「微调方向」。已有模型的能力不会被推翻,只需要在几个维度上做调整。

&nbsp;

---

## 10.2 LoRA 数学原理

LoRA 的核心公式:

$$W' = W + \Delta W = W + B \cdot A$$

其中:
- $W \in \mathbb{R}^{d \times d}$ —— 原始权重(冻结,不更新)
- $A \in \mathbb{R}^{r \times d}$ —— 降维矩阵($d \to r$)
- $B \in \mathbb{R}^{d \times r}$ —— 升维矩阵($r \to d$)
- $r$ —— rank(秩),$r \ll d$

前向传播变成:

$$h' = W'x = Wx + BAx$$

即**原始输出 + LoRA 修正项**。

**参数量对比:**
- 原始:$d \times d = d^2$
- LoRA:$d \times r + r \times d = 2rd$
- 当 $r=16, d=768$ 时:$2 \times 16 \times 768 = 24{,}576 \ll 768^2 = 589{,}824$

下面用代码验证这个数学分解:

In [ ]:
d = 768
r = 16

# 原始权重 W (frozen, 不参与训练)
W = torch.randn(d, d)
print(f"W shape: {W.shape}, 参数量: {W.numel():,}")

# LoRA 分解:ΔW = B @ A
A = torch.randn(r, d)  # (r, d) — 降维: 768 → 16
B = torch.randn(d, r)  # (d, r) — 升维: 16 → 768
print(f"A shape: {A.shape}, 参数量: {A.numel():,}")
print(f"B shape: {B.shape}, 参数量: {B.numel():,}")

# ΔW = B @ A 的 shape 是 (d, d) — 和 W 一样大!
delta_W = B @ A
print(f"\nΔW = B @ A shape: {delta_W.shape}")  # (768, 768)
print(f"ΔW 参数量(实际存储): {A.numel() + B.numel():,}")  # 24,576
print(f"W 参数量: {W.numel():,}")  # 589,824
print(f"压缩比: {W.numel() / (A.numel() + B.numel()):.1f}×")

# 前向: h' = Wx + BAx = (W + BA)x
x = torch.randn(1, d)
h_full = (W + delta_W) @ x.T          # 显式构造 W'
h_lora = W @ x.T + B @ (A @ x.T)      # 分两步(LoRA 的做法)
print(f"\n两种算法 max diff: {(h_full - h_lora).abs().max().item():.10f}")
# 完全一致!LoRA 不构造完整的 W',而是分两步算

关键洞察:LoRA **永远不显式构造** $\Delta W = BA$(那会是 $d \times d$ 的大矩阵)。它分两步算:$x \xrightarrow{A} r \xrightarrow{B} d$。中间维度只有 $r=16$,计算量极小。

> 这就是「低秩」的含义:虽然 $\Delta W$ 形式上是 $d \times d$,但它的**秩**最多为 $r$。LoRA 用 $A$ 和 $B$ 两个小矩阵来「隐式」表示这个低秩矩阵。

### 矩阵乘法的计算效率

从计算量角度看,LoRA 的前向传播只需要两次小矩阵乘法:

$$\text{FLOPs} = \underbrace{d \times r}_{A \cdot x} + \underbrace{r \times d}_{B \cdot (Ax)} = 2rd$$

对比直接用 $\Delta W$:$d \times d = d^2$。当 $r=16, d=768$ 时,$2rd = 24{,}576 \ll d^2 = 589{,}824$,计算量也少了 24 倍。

而且 LoRA 还可以做 **merge**(合并):训练完把 $BA$ 加到 $W$ 里,推理时零开销。这是 10.7 节的主题。

&nbsp;

---

## 10.3 naive LoRA(naive 版)

理解了原理,现在从零手写一个最简单的 LoRA layer。

**naive 版的设计目标**:封装 `original + lora(x)` 的 forward,让 LoRA 像一个插件一样挂到任何 Linear 层上。

In [ ]:
class NaiveLoRALinear(nn.Module):
    """
    最朴素的 LoRA 实现:
    - original: 原始 Linear 层(冻结)
    - lora_A, lora_B: 低秩矩阵(可训练)
    - forward: original(x) + B(A(x))
    """
    def __init__(self, original_linear, rank=16):
        super().__init__()
        # 保留原始 Linear(权重将被冻结)
        self.original = original_linear
        d_in = original_linear.in_features
        d_out = original_linear.out_features

        # LoRA 矩阵
        self.lora_A = nn.Linear(d_in, rank, bias=False)   # d → r
        self.lora_B = nn.Linear(rank, d_out, bias=False)  # r → d

    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x))


# 测试:用一个 768×768 的 Linear 作为 original
original = nn.Linear(768, 768, bias=False)
lora_layer = NaiveLoRALinear(original, rank=16)

# 参数量对比
orig_params = sum(p.numel() for p in original.parameters())
lora_only = sum(p.numel() for n, p in lora_layer.named_parameters() if 'lora' in n)
total = sum(p.numel() for p in lora_layer.parameters())

print(f"原始 Linear 参数量:  {orig_params:,}")
print(f"LoRA 矩阵参数量:     {lora_only:,}  (A: {768*16:,} + B: {16*768:,})")
print(f"NaiveLoRA 总参数量:  {total:,}  (含冻结的原始权重)")
print(f"可训练占比:          {lora_only / orig_params * 100:.2f}%")

# forward 验证
x = torch.randn(1, 4, 768)
out = lora_layer(x)
print(f"\n输入 shape: {x.shape}")
print(f"输出 shape: {out.shape}")  # (1, 4, 768) — 和原始 Linear 一样

naive 版工作正常。但它有个问题:**lora_A 和 lora_B 都是随机初始化的**,所以 `B(A(x))` 在训练开始时就不是零 —— 这意味着加了 LoRA 后,模型的输出立刻就变了!

这不是我们想要的。我们希望 LoRA 在**训练开始时完全等价于原始模型**,然后随着训练逐渐「学到」修正。

> naive 版有两个问题:
> 1. **初始化破坏模型**:随机初始化让 $\Delta W \neq 0$,模型输出立刻被扰动
> 2. **耦合度高**:`NaiveLoRALinear` 替换了整个 Linear 层,如果要给已有模型加 LoRA,需要手动替换每一个 Linear —— 不够灵活
>
> 10.5 节会看到 minimind 如何用 monkey-patch 解决第二个问题。现在先解决第一个。

这就需要零初始化技巧。

&nbsp;

---

## 10.4 零初始化技巧

LoRA 论文的核心技巧:**B 初始化为零矩阵,A 用高斯初始化**。

$$\Delta W = B \cdot A = \mathbf{0} \cdot A = \mathbf{0}$$

训练开始时 $\Delta W = 0$,模型行为和原始模型**完全一致**。随着训练,B 逐渐偏离零,LoRA 的修正逐渐生效。

下面在 naive 版上实现这个技巧:

In [ ]:
class NaiveLoRALinearV2(nn.Module):
    """带零初始化的 naive LoRA"""
    def __init__(self, original_linear, rank=16):
        super().__init__()
        self.original = original_linear
        d_in = original_linear.in_features
        d_out = original_linear.out_features

        self.lora_A = nn.Linear(d_in, rank, bias=False)
        self.lora_B = nn.Linear(rank, d_out, bias=False)

        # ★ 关键:A 用高斯初始化,B 用零初始化
        nn.init.normal_(self.lora_A.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x))


# 验证:加了 LoRA 后,输出应该和原始模型完全一致
torch.manual_seed(42)
original = nn.Linear(768, 768, bias=False)
lora_layer = NaiveLoRALinearV2(original, rank=16)

x = torch.randn(1, 4, 768)
with torch.no_grad():
    orig_out = original(x)          # 原始输出
    lora_out = lora_layer(x)        # 加了 LoRA 的输出

diff = (orig_out - lora_out).abs().max().item()
print(f"=== 零初始化验证 ===")
print(f"lora_B 全零: {(lora_layer.lora_B.weight == 0).all().item()}")
print(f"lora_A 均值: {lora_layer.lora_A.weight.mean().item():.6f}")
print(f"lora_A 标准差: {lora_layer.lora_A.weight.std().item():.6f}")
print(f"原始输出 vs LoRA输出 max diff: {diff}")  # 0.0!

# 显式验证 ΔW = B@A = 0
delta_W = lora_layer.lora_B.weight.data @ lora_layer.lora_A.weight.data
print(f"\nΔW = B @ A 的 max abs: {delta_W.abs().max().item()}")
print(f"→ ΔW = 0,因为 B = 0")

零初始化的效果:**训练开始时,LoRA 对模型输出没有任何影响**。梯度可以从 B 的零值开始流动,逐渐学到有用的修正。

### 为什么不把 A 也初始化为零?

如果 A 和 B 都是零,那么 $\Delta W = 0$ 且梯度 $\frac{\partial L}{\partial B} = A^T \cdot \frac{\partial L}{\partial \Delta W} = 0$ —— **梯度也是零,LoRA 永远不会开始学习**。

所以 A 必须非零(用高斯初始化),这样梯度才能从 A 传到 B,LoRA 才能开始训练。

> 这是一个精妙的设计:A 提供「方向」,B 提供「幅度」。训练开始时幅度为零(B=0),方向已就绪(A≠0),梯度让幅度逐渐增长。

&nbsp;

---

## 10.5 minimind 的 LoRA 实现

理解了 naive 版,现在看 minimind 的真实实现。打开 `model/model_lora.py`:

```python
# model_lora.py:6-18
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)   # d → r
        self.B = nn.Linear(rank, out_features, bias=False)  # r → d
        self.A.weight.data.normal_(mean=0.0, std=0.02)      # A: 高斯
        self.B.weight.data.zero_()                           # B: 零

    def forward(self, x):
        return self.B(self.A(x))
```

和我们的 naive 版几乎一样!区别是 minimind 把 LoRA 做成了一个**独立模块**(不包裹 original),只负责算 $\Delta W \cdot x = B(A(x))$。

下面用 minimind 的 `LoRA` 类做同样的验证:

In [ ]:
from model.model_lora import LoRA

# minimind 的 LoRA 模块
lora = LoRA(in_features=768, out_features=768, rank=16)

print(f"A weight shape: {lora.A.weight.shape}")  # (16, 768) — r×d
print(f"B weight shape: {lora.B.weight.shape}")  # (768, 16) — d×r
print(f"A 标准差: {lora.A.weight.std().item():.4f}")  # ~0.02
print(f"B 全零: {(lora.B.weight == 0).all().item()}")  # True

# 验证 ΔW = 0 at init
delta = lora.B.weight.data @ lora.A.weight.data
print(f"\nΔW = B @ A max abs: {delta.abs().max().item()}")  # 0.0

# forward: B(A(x)) 在 init 时应该全零
x = torch.randn(1, 4, 768)
out = lora(x)
print(f"LoRA forward output max abs: {out.abs().max().item()}")  # 0.0

# 参数量
params = sum(p.numel() for p in lora.parameters())
print(f"\nLoRA 参数量: {params:,} (A: {768*16:,} + B: {16*768:,})")

### naive 版 vs minimind 版的区别

| 方面 | naive 版 (`NaiveLoRALinearV2`) | minimind 版 (`LoRA` + `apply_lora`) |
|---|---|---|
| 结构 | 包裹原始 Linear(持有 `self.original`) | 独立模块,不持有原始层 |
| 挂载 | 手动替换每个 Linear 实例 | `apply_lora` 自动遍历+monkey-patch |
| forward | `self.original(x) + B(A(x))` | monkey-patch 后:`original_forward(x) + lora(x)` |
| 移除 | 需手动恢复原始层 | 只需删除 `.lora` 属性 + 恢复 forward |

minimind 的设计更灵活:LoRA 是一个**可插拔**的模块,通过 monkey-patch 挂载,不需要修改原始模型结构。

### apply_lora:monkey-patch 的魔法

naive 版的做法是用一个新类 `NaiveLoRALinear` 替换原始 Linear。但 minimind 用了更轻量的方式:**monkey-patch** —— 不替换类,只替换 `forward` 方法。

```python
# model_lora.py:21-32
def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.in_features == module.out_features:
            lora = LoRA(module.in_features, module.out_features, rank=rank).to(model.device)
            setattr(module, "lora", lora)              # 挂一个 lora 属性
            original_forward = module.forward          # 保存原始 forward

            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)           # 原始 + LoRA

            module.forward = forward_with_lora          # 替换 forward!
```

**monkey-patch 的原理:**
1. 遍历模型所有模块
2. 对符合条件的 Linear,**不改类、不改权重**,只加一个 `.lora` 属性
3. 把 `module.forward` 替换成 `forward_with_lora`(调用原始 forward + lora)
4. 原始权重 $W$ 原封不动,只是 forward 时多加了一项

下面用代码看 apply_lora 前后的变化:

In [ ]:
from model.model_lora import apply_lora

# apply_lora 之前:统计所有 Linear 层
linear_before = [(name, m) for name, m in model.named_modules() if isinstance(m, nn.Linear)]
print(f"=== apply_lora 之前 ===")
print(f"nn.Linear 总数: {len(linear_before)}")
total_before = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_before:,} ({total_before/1e6:.2f}M)")

# 应用 LoRA
apply_lora(model, rank=16)

# apply_lora 之后:看哪些模块被挂了 lora
lora_modules = []
for name, module in model.named_modules():
    if hasattr(module, 'lora'):
        lora_modules.append((name, module.in_features, module.out_features))

print(f"\n=== apply_lora 之后 ===")
print(f"被挂 LoRA 的模块数: {len(lora_modules)}")
total_after = sum(p.numel() for p in model.parameters())
lora_params = sum(p.numel() for n, p in model.named_parameters() if 'lora' in n)
print(f"模型参数量: {total_after:,} ({total_after/1e6:.2f}M)")
print(f"LoRA 参数量: {lora_params:,} ({lora_params/1e6:.4f}M)")
print(f"LoRA 占比: {lora_params / total_after * 100:.2f}%")

### monkey-patch 的闭包陷阱

注意 `forward_with_lora` 的参数绑定:

```python
def forward_with_lora(x, layer1=original_forward, layer2=lora):
    return layer1(x) + layer2(x)
```

为什么用默认参数 `layer1=original_forward, layer2=lora` 而不是直接用闭包捕获?

因为 Python 闭包是**延迟绑定**的 —— 如果直接写 `return original_forward(x) + lora(x)`,在循环中 `original_forward` 和 `lora` 会指向**最后一个循环迭代的值**,而不是当前迭代的值。用默认参数在函数定义时就绑定了正确的值,这是 Python 中常见的闭包陷阱解决方案。

> 这个技巧在 Python 中非常常见:把循环变量作为默认参数传入,确保每个函数捕获的是自己那一轮的值。

### 关键设计决策:`in_features == out_features` 过滤

`apply_lora` 最重要的一行是过滤条件:

```python
if isinstance(module, nn.Linear) and module.in_features == module.out_features:
```

只给**方形 Linear**(输入维度 = 输出维度)挂 LoRA。下面看看 minimind 模型里哪些层满足这个条件:

In [ ]:
# 遍历所有 Linear,标注是否被挂 LoRA
print(f"{'模块名':<45s} {'in':>5s} {'out':>5s} {'方形':>4s} {'挂LoRA':>6s}")
print("=" * 70)

seen = set()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        short = name.replace(f"model.layers.", "L")
        key = short.split('.')[0] + '.' + short.split('.')[-1] if '.' in short else short
        # 只显示 layer 0 的(其他层一样)
        layer_num = name.split('.')[2] if 'layers' in name else '-'
        if layer_num == '0' or 'lm_head' in name:
            square = module.in_features == module.out_features
            has_lora = hasattr(module, 'lora')
            print(f"{name:<45s} {module.in_features:>5d} {module.out_features:>5d} "
                  f"{'✓' if square else '✗':>4s} {'✓' if has_lora else '✗':>6s}")

print("\n(仅展示 layer 0,其他 7 层结构相同)")

结果一目了然:

| 模块 | 维度 | 方形 | 挂 LoRA |
|---|---|---|---|
| `q_proj` | 768→768 | ✓ | ✓ |
| `k_proj` | 768→**384** | ✗ | ✗ |
| `v_proj` | 768→**384** | ✗ | ✗ |
| `o_proj` | 768→768 | ✓ | ✓ |
| `gate_proj` | 768→2432 | ✗ | ✗ |
| `down_proj` | 2432→768 | ✗ | ✗ |
| `up_proj` | 768→2432 | ✗ | ✗ |
| `lm_head` | 768→6400 | ✗ | ✗ |

**为什么 k/v_proj 不被挂?** 因为 minimind 使用 **GQA(Grouped Query Attention)**:`num_key_value_heads=4 < num_attention_heads=8`,所以 k/v_proj 的输出维度是 `4 × 96 = 384`,不等于输入的 768。

**q_proj 为什么是方形的?** 因为 `num_attention_heads × head_dim = 8 × 96 = 768 = hidden_size`。这是标准 Transformer 的设计 —— q_proj 把 hidden_size 映射到所有 head 的总维度。

> 这个过滤条件是一个**刻意的简化**:不单独处理非方形情况,用「方形」作为天然的筛选器,顺便利用了 GQA 的特性。结果只有 q_proj 和 o_proj 被挂 LoRA,FFN 和 lm_head 全部跳过。

In [ ]:
# 精确列出被挂 LoRA 的模块
print("=== 被挂 LoRA 的模块(全部 8 层)===")
for name, in_f, out_f in lora_modules:
    print(f"  {name:<45s} {in_f}→{out_f}")

print(f"\n共 {len(lora_modules)} 个模块")
print(f"每层: q_proj + o_proj = 2 个")
print(f"8 层: 2 × 8 = {len(lora_modules)} 个")
print(f"\n每个模块 LoRA 参数: A(768×16) + B(16×768) = {768*16 + 16*768:,}")
print(f"总计: {768*16 + 16*768} × {len(lora_modules)} = {(768*16 + 16*768) * len(lora_modules):,}")

### monkey-patch 后的 forward 验证

验证替换 forward 后,模型在零初始化时行为不变:

In [ ]:
# 重建模型(干净的)
model_clean = MiniMindForCausalLM(MiniMindConfig()).eval()

input_ids = torch.tensor([[1, 5310, 2863, 2]])

# 原始模型输出
with torch.no_grad():
    orig_out = model_clean(input_ids).logits.clone()

# apply_lora(B=0 → delta=0 → 输出应该不变)
apply_lora(model_clean, rank=16)

with torch.no_grad():
    lora_out = model_clean(input_ids).logits.clone()

diff = (orig_out - lora_out).abs().max().item()
print(f"=== monkey-patch forward 验证 ===")
print(f"apply_lora 前后 max diff: {diff}")
print(f"→ 零初始化确保 LoRA 对模型输出零影响")

# 检查 forward 是否被替换了
q_proj = model_clean.model.layers[0].self_attn.q_proj
fwd_name = q_proj.forward.__name__ if hasattr(q_proj.forward, '__name__') else 'unknown'
print(f"\nq_proj.forward 函数名: {fwd_name}")
print(f"q_proj 有 lora 属性: {hasattr(q_proj, 'lora')}")

### monkey-patch 小结

`apply_lora` 做了三件事:
1. **创建** LoRA 模块(`setattr(module, "lora", lora)`)
2. **保存** 原始 forward(`original_forward = module.forward`)
3. **替换** forward(`module.forward = forward_with_lora`)

整个过程**不改原始权重、不改类定义**,只在运行时修改实例的 `forward` 方法。这是 Python 的动态特性带来的灵活性 —— 代价是不能和 `torch.compile` 配合(10.6 节)。

> monkey-patch 的本质:Python 中方法只是对象的属性。替换 `module.forward` 就等于替换了这个模块的行为,而原始类定义(`nn.Linear`)完全不受影响。

&nbsp;

---

## 10.6 LoRA 训练

打开 `trainer/train_lora.py`,看 LoRA 训练和全参 SFT 的关键区别。

### 1. 冻结所有非 LoRA 参数

```python
# train_lora.py:139-146
lora_params = []
for name, param in model.named_parameters():
    if 'lora' in name:
        param.requires_grad = True       # LoRA 参数:可训练
        lora_params.append(param)
    else:
        param.requires_grad = False      # 其他参数:冻结!
```

全参 SFT 里所有参数的 `requires_grad=True`;LoRA 里只有名字含 `'lora'` 的参数才训练。

### 2. optimizer 只接收 lora_params

```python
# train_lora.py:152
optimizer = optim.AdamW(lora_params, lr=args.learning_rate)
```

全参 SFT 是 `optim.AdamW(model.parameters(), ...)`,优化器要维护**全量**参数的 momentum + variance。

LoRA 只优化 `lora_params`(0.39M),optimizer state 缩小了 **160 倍**。

### 3. 学习率高 10 倍

```python
# train_lora.py:84
parser.add_argument("--learning_rate", type=float, default=1e-4)
```

对比:

| 训练阶段 | 默认 lr | 说明 |
|---|---|---|
| Pretrain | 5e-4 | 从头训练,需要大步长 |
| Full SFT | 1e-5 | 微调已训练好的模型,小步长避免破坏 |
| **LoRA** | **1e-4** | 比 SFT 高 10 倍 |

LoRA 只训练 0.6% 的参数,梯度信号集中在这些参数上,需要更大的步长才能有效学习。

> 这也解释了为什么 LoRA 的 epochs=10 比 SFT 的 3-5 更多:可训练参数少,每个 epoch 能学到的信息也少,需要更多轮次来弥补。

下面用代码模拟训练流程:

In [ ]:
# === 模拟 LoRA 训练的参数冻结 ===
model_train = MiniMindForCausalLM(MiniMindConfig())
apply_lora(model_train, rank=16)

# 冻结非 LoRA 参数
lora_params = []
frozen_params = []
for name, param in model_train.named_parameters():
    if 'lora' in name:
        param.requires_grad = True
        lora_params.append(param)
    else:
        param.requires_grad = False
        frozen_params.append(param)

trainable = sum(p.numel() for p in model_train.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model_train.parameters() if not p.requires_grad)
total = trainable + frozen

print(f"=== LoRA 训练参数统计 ===")
print(f"可训练(LoRA):   {trainable:>12,} ({trainable/1e6:.4f}M)")
print(f"冻结(原始):     {frozen:>12,} ({frozen/1e6:.2f}M)")
print(f"总计:            {total:>12,} ({total/1e6:.2f}M)")
print(f"可训练占比:      {trainable/total*100:.2f}%")

# optimizer 只看 lora_params
optimizer = torch.optim.AdamW(lora_params, lr=1e-4)
print(f"\noptimizer param groups: {len(optimizer.param_groups)}")
print(f"optimizer 管理参数数: {sum(len(g['params']) for g in optimizer.param_groups)}")
print(f"学习率: {optimizer.param_groups[0]['lr']}")

### 4. torch.compile 被禁用

```python
# train_lora.py:164-166
if args.use_compile == 1:
    args.use_compile = 0
    Logger('[LoRA] monkey-patch forward 与 torch.compile 不兼容，use_compile 已自动关闭')
```

monkey-patch 替换了 `module.forward`,而 `torch.compile` 需要静态的 forward 图来编译优化。两者冲突 —— `torch.compile` 会尝试追踪被 patch 的 forward,结果不可预测。

所以 LoRA 训练**强制关闭** `torch.compile`,即使用户传了 `--use_compile 1`。

> `torch.compile` 在训练开始前编译模型的 forward 图。monkey-patch 替换了 `forward` 属性(从 bound method 变成普通函数),`torch.compile` 追踪到的是一个动态绑定的函数而非静态图,编译结果不可预测。这是一个已知的兼容性限制。

### 5. 训练配置总结

```python
# train_lora.py 的关键默认值
defaults = {
    'epochs': 10,                    # 比 SFT 多(全参 SFT 通常 3-5 epochs)
    'learning_rate': 1e-4,          # SFT 的 10 倍
    'batch_size': 32,
    'from_weight': 'full_sft',      # 必须基于已 SFT 的模型
    'data_path': '../dataset/lora_medical.jsonl',
    'use_compile': 0,               # 强制关闭
}
for k, v in defaults.items():
    print(f"  {k:<20s} = {v}")

In [ ]:
print("=== train_lora.py 关键默认配置 ===")
defaults = {
    'epochs': 10,
    'learning_rate': 1e-4,
    'batch_size': 32,
    'from_weight': 'full_sft',
    'data_path': '../dataset/lora_medical.jsonl',
    'use_compile': 0,
}
for k, v in defaults.items():
    print(f"  {k:<20s} = {v}")

print("\n=== lr 对比 ===")
print(f"  Pretrain lr: 5e-4")
print(f"  Full SFT lr: 1e-5")
print(f"  LoRA lr:     1e-4  (= SFT × {1e-4/1e-5:.0f})")

> LoRA 训练必须基于 `from_weight='full_sft'` —— 即一个已经完成 SFT 的模型。LoRA 是在已有能力上做**领域适配**(如医学问答),不是从头训练。

### 训练数据:`lora_medical.jsonl`

LoRA 的训练数据和 SFT 格式完全一样(ChatML 风格的 JSONL),但内容是**特定领域**的问答对。minimind 默认用 `lora_medical.jsonl` —— 医学问答数据。

训练时只从 assistant 的回复部分计算 loss(prompt 部分标为 -100,第 9 章详解),让模型学会医学领域的回答风格和知识。

> 换一个领域(如法律、编程),只需准备新的 JSONL 文件,改 `--lora_name` 和 `--data_path`,就能训出一个新的 LoRA。base model 不变,每个 LoRA 只需 ~0.8 MB。

&nbsp;

---

## 10.7 保存与合并

LoRA 的另一个优势:**只存 LoRA 权重**,文件极小。

### save_lora:只存补丁

```python
# model_lora.py:45-53
def save_lora(model, path):
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {}
    for name, module in raw_model.named_modules():
        if hasattr(module, 'lora'):
            clean_name = name[7:] if name.startswith("module.") else name
            lora_state = {f'{clean_name}.lora.{k}': v.cpu().half()
                          for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)
```

只遍历有 `.lora` 属性的模块,只存 `A.weight` 和 `B.weight`。原始权重完全不存(已经有了)。

### merge_lora:把补丁折叠进原始权重

训练完之后,可以把 LoRA 的 $\Delta W = BA$ **加到**原始权重里,得到一个不需要 LoRA 的完整模型:

$$W_{\text{merged}} = W + B \cdot A$$

合并后丢弃 LoRA,推理时不再需要额外的 forward 开销。

```python
# model_lora.py:56-65
def merge_lora(model, lora_path, save_path):
    load_lora(model, lora_path)
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {k: v.cpu().half() for k, v in raw_model.state_dict().items()
                  if '.lora.' not in k}
    for name, module in raw_model.named_modules():
        if isinstance(module, nn.Linear) and '.lora.' not in name:
            state_dict[f'{name}.weight'] = module.weight.data.clone().cpu().half()
            if hasattr(module, 'lora'):
                state_dict[f'{name}.weight'] += (module.lora.B.weight.data @ module.lora.A.weight.data).cpu().half()
    torch.save(state_dict, save_path)
```

下面完整演示 **save → load → merge** 流程:

In [ ]:
from model.model_lora import apply_lora, save_lora, load_lora, merge_lora
import tempfile, os

# 准备模型 + LoRA
model_test = MiniMindForCausalLM(MiniMindConfig()).eval()
apply_lora(model_test, rank=16)

# 保存 base model 权重(用于后续验证 load_lora)
base_state = {k: v.clone() for k, v in model_test.state_dict().items() if 'lora' not in k}

# 模拟「训练」:给 LoRA 的 B 矩阵加上随机值(假装训练过了)
torch.manual_seed(42)
for name, module in model_test.named_modules():
    if hasattr(module, 'lora'):
        module.lora.B.weight.data += torch.randn_like(module.lora.B.weight.data) * 0.01

input_ids = torch.tensor([[1, 5310, 2863, 2]])

# 训练后(有 LoRA 修正)的输出
with torch.no_grad():
    trained_out = model_test(input_ids).logits.clone()

# === Step 1: save_lora(只存 LoRA 权重)===
tmpdir = tempfile.mkdtemp()
lora_path = os.path.join(tmpdir, 'lora_medical.pth')
save_lora(model_test, lora_path)

lora_file_size = os.path.getsize(lora_path)
print(f"=== Step 1: save_lora ===")
print(f"保存路径: {lora_path}")
print(f"文件大小: {lora_file_size:,} bytes ({lora_file_size/1024:.1f} KB)")

# 看存了什么
saved_sd = torch.load(lora_path, map_location='cpu')
print(f"保存的 key 数: {len(saved_sd)}")
print(f"前 4 个 key:")
for k in list(saved_sd.keys())[:4]:
    print(f"  {k}: {saved_sd[k].shape}")

778.7 KB —— 这就是 LoRA 的全部!对比全参 SFT 的模型文件(131 MB),**小了 168 倍**。

你可以为不同领域训练不同的 LoRA,每个只需 ~0.8 MB,切换领域只需加载一个小文件。

### save_lora 存了什么?

保存的 key 格式是 `{module_name}.lora.{A|B}.weight`。每个被挂 LoRA 的模块有 2 个 key(A 和 B),16 个模块共 32 个 key。

注意保存时用 `.cpu().half()` —— LoRA 权重转成 fp16 存盘,进一步压缩文件大小。对于 393,216 个参数,fp32 需要 ~1.5 MB,fp16 只需 ~0.8 MB。

In [ ]:
# === Step 2: load_lora(加载到新模型)===
# 模拟实际使用:加载 base model,再加载 LoRA
model_base = MiniMindForCausalLM(MiniMindConfig()).eval()
# 加载相同的 base 权重(确保和 model_test 的基础模型一致)
model_base.load_state_dict(base_state, strict=False)
apply_lora(model_base, rank=16)  # 先挂 LoRA 结构(B=0)

# 加载保存的 LoRA 权重
load_lora(model_base, lora_path)

# 验证加载后的输出和训练后一致
with torch.no_grad():
    loaded_out = model_base(input_ids).logits.clone()

diff = (trained_out - loaded_out).abs().max().item()
print(f"=== Step 2: load_lora ===")
print(f"训练后 vs 加载后 max diff: {diff}")  # 应该非常小(fp16 精度)
print(f"→ LoRA 权重正确恢复")

### load_lora 的加载过程

`load_lora` 先用 `apply_lora` 挂载空的 LoRA 结构(B=0),然后从文件中读取训练好的 A、B 权重,用 `module.lora.load_state_dict()` 恢复。

注意 `load_lora` 的 key 匹配逻辑:它遍历所有有 `.lora` 属性的模块,从 state_dict 中筛选出匹配的 key(如 `model.layers.0.self_attn.q_proj.lora.A.weight`),去掉前缀后加载到对应模块。

> 加载时需要 rank 一致:`save_lora` 时 rank=16,`load_lora` 时也必须先 `apply_lora(model, rank=16)`。rank 不匹配会导致 A/B 维度不兼容。

In [ ]:
# === Step 3: merge_lora(折叠进原始权重)===
merged_path = os.path.join(tmpdir, 'merged_model.pth')
merge_lora(model_base, lora_path, merged_path)

merged_file_size = os.path.getsize(merged_path)
print(f"=== Step 3: merge_lora ===")
print(f"保存路径: {merged_path}")
print(f"文件大小: {merged_file_size:,} bytes ({merged_file_size/1024/1024:.1f} MB)")

# 加载 merged 模型(不需要 LoRA)
model_merged = MiniMindForCausalLM(MiniMindConfig()).eval()
model_merged.load_state_dict(torch.load(merged_path, map_location='cpu'))

# 验证 merged 输出和 LoRA 模型一致
with torch.no_grad():
    merged_out = model_merged(input_ids).logits.clone()

diff = (loaded_out - merged_out).abs().max().item()
print(f"\nLoRA 模型 vs merged 模型 max diff: {diff}")
print(f"→ merge 后无需 LoRA,输出一致(fp16 精度误差)")

# merged 模型没有 lora 属性
has_lora = any(hasattr(m, 'lora') for m in model_merged.modules())
print(f"\nmerged 模型还有 lora 属性吗: {has_lora}")  # False
print(f"→ merge 后 LoRA 被完全吸收")

### merge 的数学本质

合并操作:对每个挂了 LoRA 的 Linear,把 $BA$ 加到原始 $W$ 上:

$$W_{\text{merged}} = W + B \cdot A$$

合并后:
- $W_{\text{merged}}$ 的 shape 和 $W$ 一样($d \times d$)
- 推理时 $h' = W_{\text{merged}} \cdot x$,不需要额外的 LoRA forward
- 丢弃 LoRA 模块,推理零额外开销

> 注意 merge 使用 fp16(`.half()`),精度有微小损失(~0.002)。对于推理质量影响可忽略,但如果需要精确复现 LoRA 模型输出,用 `load_lora` 而非 `merge_lora`。

### save / load / merge 的使用场景

| 操作 | 用途 | 文件大小 |
|---|---|---|
| `save_lora` | 训练完保存 LoRA 权重 | ~0.8 MB |
| `load_lora` | 加载 base model + LoRA 做推理 | 两个文件 |
| `merge_lora` | 把 LoRA 折叠进 base model | ~131 MB(完整模型) |

**推理时的选择:**
- 如果要**频繁切换领域**(如多轮对话中切换医学/法律/编程)→ 用 `load_lora`,切换只需换 LoRA 文件
- 如果是**单领域部署** → 用 `merge_lora`,合并后无额外开销

> merge 后的精度误差来自 fp16 half precision。`merge_lora` 里所有计算和存储都用 `.half()`,合并后精度损失约 0.002,对推理质量影响可忽略。

&nbsp;

---

## 10.8 LoRA vs 全参 SFT 对比

把两种微调方式做一个全面对比:

In [ ]:
# === 参数量对比 ===
model_full = MiniMindForCausalLM(MiniMindConfig())
model_lora_version = MiniMindForCausalLM(MiniMindConfig())
apply_lora(model_lora_version, rank=16)

# 全参 SFT:所有参数可训练
full_trainable = sum(p.numel() for p in model_full.parameters())

# LoRA:只有 lora 参数可训练
lora_trainable = sum(p.numel() for n, p in model_lora_version.named_parameters() if 'lora' in n)

# Optimizer state (AdamW: 2× trainable params for momentum + variance)
full_opt_state = full_trainable * 2  # fp32: momentum + variance
lora_opt_state = lora_trainable * 2

print(f"{'指标':<30s} {'全参 SFT':>15s} {'LoRA':>15s} {'比值':>10s}")
print("=" * 75)
print(f"{'可训练参数':<30s} {full_trainable:>15,} {lora_trainable:>15,} {full_trainable/lora_trainable:>9.0f}×")
print(f"{'可训练参数 (M)':<30s} {full_trainable/1e6:>14.2f}M {lora_trainable/1e6:>14.4f}M")
print(f"{'Optimizer state (M params)':<30s} {full_opt_state/1e6:>14.2f}M {lora_opt_state/1e6:>14.4f}M")
print(f"{'梯度 (M params)':<30s} {full_trainable/1e6:>14.2f}M {lora_trainable/1e6:>14.4f}M")
print(f"{'可训练占比':<30s} {'100.00%':>15s} {lora_trainable/(full_trainable+lora_trainable)*100:>14.2f}%")
print(f"{'保存文件大小':<30s} {'~131 MB':>15s} {'~0.8 MB':>15s} {'163×':>10s}")
print(f"{'默认学习率':<30s} {'1e-5':>15s} {'1e-4':>15s} {'10×':>10s}")
print(f"{'默认 epochs':<30s} {'3-5':>15s} {'10':>15s}")
print(f"{'torch.compile':<30s} {'支持':>15s} {'不支持':>15s}")

### 显存对比

上面的数字看起来很抽象,换成显存更直观:

| 内存项 | 全参 SFT (fp32) | LoRA (fp32) |
|---|---|---|
| 模型权重 | 256 MB | 256 MB(冻结,但仍在显存) |
| 梯度 | 256 MB | 1.6 MB(只有 LoRA) |
| Optimizer state | 512 MB | 3.1 MB |
| **总计** | **~1 GB** | **~261 MB** |

LoRA 把训练时的额外显存(梯度 + optimizer)从 768 MB 降到 4.7 MB —— **少了 160 倍**。这就是为什么 LoRA 能在消费级 GPU 上微调大模型。

> 注意:冻结的原始权重仍需占用显存。LoRA 省的是「可训练部分的额外开销」,不是模型本身。

### 什么时候用 LoRA vs 全参?

| 场景 | 推荐 | 理由 |
|---|---|---|
| **领域适配**(医学、法律、客服) | LoRA | 注入领域知识,不改变基础能力 |
| **多领域切换** | LoRA | 每个领域一个小文件,热切换 |
| **显存不够** | LoRA | optimizer state 缩小 160× |
| **快速实验** | LoRA | 训练快,迭代快 |
| **基础能力提升**(推理、代码) | 全参 SFT | 需要深度改变模型行为 |
| **风格/格式微调** | 全参 SFT 或 LoRA | 轻微改变可用 LoRA,深度改变需全参 |
| **首次微调** | 全参 SFT | 从 pretrain 到 SFT 是质变,建议全参 |

> LoRA 的哲学是「**小补丁、大效果**」。它假设微调时的权重变化是低秩的 —— 这个假设在大多数领域适配场景下成立。但如果微调需要改变模型的「基础推理方式」,低秩假设可能不够,全参更合适。

### 实际建议

对于个人开发者和小团队:

1. **先用 LoRA 试水** —— 训练快、成本低,效果不好再换全参
2. **rank 选择**:minimind 用 16,大模型(7B+)常用 8-64。rank 越大效果越好但参数越多
3. **多 LoRA 叠加**:可以同时加载多个 LoRA(不同领域),按需切换或叠加
4. **merge 后部署**:生产环境用 merge 后的模型,推理零开销

> LoRA 不是全参 SFT 的「廉价替代品」,而是一种**不同微调范式**:适合领域注入,不适合基础能力改造。理解两者的适用场景,是高效训练的关键。

&nbsp;

---

## Summary and takeaways

### 1. LoRA 的核心思想

$$W' = W + \Delta W = W + B \cdot A, \quad A \in \mathbb{R}^{r \times d}, \; B \in \mathbb{R}^{d \times r}, \; r \ll d$$

冻结 $W$,只训练 $A$ 和 $B$。参数量从 $d^2$ 降到 $2rd$。

### 2. 零初始化:训练起点 = 原始模型

$B = \mathbf{0} \Rightarrow \Delta W = \mathbf{0}$。训练开始时 LoRA 不改变模型输出,随着训练逐渐生效。

### 3. minimind 的 monkey-patch 实现

- `apply_lora`:不替换类,只替换 `module.forward`
- 过滤 `in_features == out_features`:挂 q_proj + o_proj(768→768),跳过 k/v_proj(768→384,GQA)
- LoRA 参数:393,216(0.39M)= 总参数的 **0.61%**

### 4. 训练差异

| | 全参 SFT | LoRA |
|---|---|---|
| 可训练参数 | 63.9M(100%) | 0.39M(0.61%) |
| lr | 1e-5 | 1e-4(10×) |
| optimizer state | 全量 | 0.61% |
| torch.compile | ✓ | ✗(与 monkey-patch 冲突) |
| 保存大小 | ~131 MB | ~0.8 MB |

### 5. 保存与合并

- `save_lora`:只存 A+B(~0.8 MB)
- `merge_lora`:$W_{\text{merged}} = W + BA$,折叠后无需 LoRA

### 最终对比:LoRA 完整画像

```
原始模型 (63.9M, 冻结)
    │
    ├── apply_lora → +LoRA (0.39M, 可训练)  ← 0.61%
    │
    ├── train → lr=1e-4, epochs=10
    │
    ├── save_lora → lora_medical.pth (~0.8 MB)
    │
    └── merge_lora → W + BA → 完整模型 (~131 MB)
```

LoRA 的三句话总结:
1. **数学**:用两个小矩阵 $A, B$ 近似权重变化 $\Delta W$,参数减少 24×
2. **初始化**:$B=0$ 保证训练从预训练模型平滑出发
3. **工程**:monkey-patch 挂载、只存补丁、可合并 —— 最小侵入式微调

> **下一步**:LoRA 只是参数高效微调的一种。在掌握 SFT 和 LoRA 之后,下一步是学习如何让模型「对齐人类偏好」 —— 用 RLHF / DPO / GRPO 等方法。这就是第 11 章的主题。
>
> → [第 11 章 · 推理工程](../ch11/01_main-chapter-code/README.md)